In [ ]:
import pandas as pd
from collections import deque
from typing import Tuple, Dict, Set, List
from collections import defaultdict


In [83]:
df_clinical = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_clinical_data_with_mondo_parents_mondo_cleaned.csv", dtype=str)
df_preclinical = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_preclinical_data_with_mondo_parents_mondo_cleaned.csv", dtype=str)


In [84]:
df_clinical.shape, df_preclinical.shape

((18609, 14), (547365, 15))

In [85]:
df_preclinical.head()

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean
0,31733831,asthma,isorhynchophylline,asthma,isorhynchophylline,asthma,MONDO:0004979,isorhynchophylline,C0245133,-1,-1,MONDO:0004979,asthma,asthma,MONDO:0004979
1,31733833,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,MONDO:0005068,antgomir-1192|mir-1192|agomir-1192,-1|-1|-1,-1,-1,MONDO:0005068,myocardial infarction,myocardial infarction,MONDO:0005068
2,31733925,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,MONDO:0007915,g2|HLA-G2 Isoform,-1|C0967254,-1,-1,MONDO:0007915,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915
3,31733940,cognitive impairment,minocycline,cognitive impairment,minocycline,cognitive disorder,MONDO:0002039,Minocycline,C0026187,-1,-1,MONDO:0002039,cognitive disorder,cognitive disorder,MONDO:0002039
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,Tadalafil|Tadalafil,C1176316|C1176316,-1,-1,-1|-1|MONDO:0003620,oxaliplatin-induced peripheral neuropathy|cumu...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620


## Create UMLS Mappings

In [5]:
umls_full = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/mrconso_filtered_db_and_sty_474316_drug_chemical_level_0_9.csv")

In [6]:
umls_full.head()

,cui,srl,sab,str,sty
0,C5687455,9,SNOMEDCT_US,Solution for infusion and/or injection,Biomedical or Dental Material
1,C5687658,9,SNOMEDCT_US,Amlodipine- and atenolol-containing product in...,Clinical Drug
2,C5688824,9,SNOMEDCT_US,Prolonged-release periodontal insert,Biomedical or Dental Material
3,C5690295,0,MSH,ACEA100610,Organic Chemical
4,C5690315,0,MSH,"MIRN3113 microRNA, human",Biologically Active Substance


In [7]:
umls_full["sty"].value_counts()


sty
Organic Chemical                           199684
Clinical Drug                               98293
Biologically Active Substance               72107
Amino Acid, Peptide, or Protein             30375
Enzyme                                      30083
Immunologic Factor                          15786
Nucleic Acid, Nucleoside, or Nucleotide      8415
Inorganic Chemical                           4639
Antibiotic                                   4436
Biomedical or Dental Material                4189
Hormone                                      2081
Element, Ion, or Isotope                     1675
Drug Delivery Device                         1362
Vitamin                                       901
Chemical Viewed Structurally                  274
Chemical                                       16
Name: count, dtype: int64

In [8]:
umls_full[
    (umls_full['sty'] == "Organic Chemical") &
    (umls_full['sab'] == "SNOMEDCT_US")
]

,cui,srl,sab,str,sty
401,C0002483,9,SNOMEDCT_US,Amidine,Organic Chemical
405,C0003420,9,SNOMEDCT_US,Antipyrine,Organic Chemical
410,C0006469,9,SNOMEDCT_US,Butadiene,Organic Chemical
413,C0006865,9,SNOMEDCT_US,Cannabinol,Organic Chemical
414,C0006931,9,SNOMEDCT_US,Capsaicin,Organic Chemical
...,...,...,...,...,...
467814,C5238148,9,SNOMEDCT_US,Foscarbidopa,Organic Chemical
467865,C5422747,9,SNOMEDCT_US,Ripasudil hydrochloride,Organic Chemical
473064,C0209435,9,SNOMEDCT_US,3-dechloroethylifosfamide,Organic Chemical
473076,C0687700,9,SNOMEDCT_US,CCNU,Organic Chemical


In [9]:
umls_full[umls_full['cui']=="C0031465"]

,cui,srl,sab,str,sty
469031,C0031465,0,MSH,Phenylbutyrates,Organic Chemical


In [10]:
cui_to_str = pd.Series(umls_full['str'].values, index=umls_full['cui']).to_dict()
cui_to_str['C5198068']

'fty720-c2'

In [11]:
cui_to_sty = pd.Series(umls_full['sty'].values, index=umls_full['cui']).to_dict()
cui_to_sty['C5198068']

'Organic Chemical'

In [12]:
mappings = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/mrrel_all_drug_rela_20251209.csv")
mappings["cui1"] = mappings["cui1"].astype(str).str.strip()
mappings["cui2"] = mappings["cui2"].astype(str).str.strip()


In [13]:
mappings.rela.unique()

array(['mapped_to', 'form_of'], dtype=object)

In [14]:
mappings.head()

,cui1,aui1,stype1,rel,cui2,aui2,stype2,rela,rui,srui,sab,sl,rg,dir,suppress,cvf,dummy
0,C0000173,A0016717,SDUI,RN,C0044909,A0140525,SDUI,mapped_to,R148187775,NaN,MSH,MSH,1.0,NaN,N,NaN,NaN
1,C0000173,A0016717,SDUI,RN,C0045030,A0140860,SDUI,mapped_to,R148219995,NaN,MSH,MSH,1.0,NaN,N,NaN,NaN
2,C0000173,A0016717,SDUI,RN,C0045031,A0140863,SDUI,mapped_to,R148193004,NaN,MSH,MSH,1.0,NaN,N,NaN,NaN
3,C0000173,A0016717,SDUI,RN,C0601554,A1326303,SDUI,mapped_to,R148155163,NaN,MSH,MSH,1.0,NaN,N,NaN,NaN
4,C0000173,A0016717,SDUI,RN,C0609759,A1326223,SDUI,mapped_to,R148209126,NaN,MSH,MSH,1.0,NaN,N,NaN,NaN


In [57]:
mappings[mappings['cui1']=="C4018784"]

,cui1,aui1,stype1,rel,cui2,aui2,stype2,rela,rui,srui,sab,sl,rg,dir,suppress,cvf,dummy
446148,C4018784,A24776656,SCUI,RN,C5835382,A35701875,SCUI,form_of,R218133869,NaN,RXNORM,RXNORM,NaN,NaN,N,NaN,NaN
446149,C4018784,A24776656,SCUI,RN,C5836039,A35710589,SCUI,form_of,R218133870,NaN,RXNORM,RXNORM,NaN,NaN,N,NaN,NaN
446150,C4018784,A24776656,SCUI,RN,C5836354,A35709910,SCUI,form_of,R217980924,NaN,RXNORM,RXNORM,NaN,NaN,N,NaN,NaN


In [16]:
cui2_to_cui1 = defaultdict(list)

# build mapping
for _, row in mappings.iterrows():
    cui1 = row['cui1'].strip()
    cui2 = row['cui2'].strip()
    if pd.notna(cui1) and pd.notna(cui2):
        cui2_to_cui1[cui2].append(cui1)

# optionally convert to a normal dict
cui2_to_cui1 = dict(cui2_to_cui1)

In [17]:
parent_to_children = defaultdict(set)

for cui2, parents in cui2_to_cui1.items():
    for cui1 in parents:
        parent_to_children[cui1].add(cui2) 

parent_counts = {parent: len(children) for parent, children in parent_to_children.items()}
top10 = sorted(parent_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for mapping, count in top10:
    if mapping in cui_to_str:
        print(f"-> maps to {cui_to_str[mapping]} {count}")

-> maps to Arabidopsis Proteins 5758
-> maps to Bacterial Proteins 4251
-> maps to Membrane Protein 3686
-> maps to Saccharomyces cerevisiae Proteins 3594
-> maps to Drosophila Proteins 3348
-> maps to Factor, Transcription 3120
-> maps to Oligopeptides 2910


In [61]:
id_to_map = "C3712411"
print(f"{id_to_map} is {cui_to_str[id_to_map]}")
for mapping in cui2_to_cui1[id_to_map]:
    print(f"-> maps to {cui_to_str[mapping]} {mapping}")
    if mapping in cui2_to_cui1:
        for deeper_map in cui2_to_cui1[mapping]:
            print(f"--> maps to {cui_to_str[deeper_map]}")

C3712411 is 177Lu-DOTA-rituximab
-> maps to Organometallic Compounds C0029252
-> maps to riTUXimab C0393022


In [63]:
parent_counts['C0393022']

15

In [19]:
def get_all_mappings_with_labels(start_cui, cui2_to_cui1, cui_to_str):
    """
    Return all CUIs reachable from `start_cui` (any depth) and their labels.

    Parameters
    ----------
    start_cui : str
        The root CUI to traverse from (will be included in `all_ids`).
    cui2_to_cui1 : dict[str, list[str]]
        Mapping dictionary where each CUI may map to one or more CUIs.
    cui_to_str : dict[str, str]
        Dictionary mapping CUI IDs to human-readable string labels.

    Returns
    -------
    all_ids : set[str]
        Set of all reachable CUIs, including `start_cui`.
    all_pairs : list[tuple[str, str]]
        List of (CUI, label) pairs for all reachable CUIs,
        **excluding the starting CUI**.
    """
    visited = set()
    stack = [start_cui]

    while stack:
        cui = stack.pop()
        if cui in visited:
            continue
        visited.add(cui)

        if cui in cui2_to_cui1:
            for nxt in cui2_to_cui1[cui]:
                if nxt not in visited:
                    stack.append(nxt)

    # Build list of (cui, label), excluding the start CUI
    all_pairs = [
        (cui, cui_to_str[cui])
        for cui in visited
        if cui != start_cui and cui in cui_to_str
    ]

    return all_pairs


In [20]:
get_all_mappings_with_labels(id_to_map, cui2_to_cui1, cui_to_str)

[('C1699926', 'Fingolimod'),
 ('C0007745', 'Ceramides'),
 ('C0388087', 'Fingolimod hydrochloride')]

In [107]:
id_to_map = "C3712411"
print(f"{id_to_map} is {cui_to_str[id_to_map]}")
for mapping in cui2_to_cui1[id_to_map]:
    print(f"-> maps to {cui_to_str[mapping]}")
    if mapping in cui2_to_cui1:
        for deeper_map in cui2_to_cui1[mapping]:
            print(f"--> maps to {cui_to_str[deeper_map]}")

C3712411 is 177Lu-DOTA-rituximab
-> maps to Organometallic Compounds
-> maps to riTUXimab


In [22]:
all_umls_ids_clinical = {
    tid
    for cell in df_clinical['drug_umls_termid']
    for tid in cell.split('|')
    if tid and tid != '-1'
}

all_umls_ids_preclinical = {
    tid
    for cell in df_preclinical['drug_umls_termid']
    for tid in cell.split('|')
    if tid and tid != '-1'
}

all_umls_ids = all_umls_ids_clinical.union(all_umls_ids_preclinical)


## Assign UMLS parent concepts

In [86]:
def assign_nearest_dataset_parents(
    df: pd.DataFrame,
    cui2_to_cui1: Dict[str, List[str]],
    cui_to_str: Dict[str, str],
    all_umls_ids: Set[str],
    parent_counts: Dict[str, int],
    id_column: str = "drug_umls_termid",
    tokens_column: str = "drug_term_umls_norm"
) -> pd.DataFrame:

    parent_ids: list[str] = []
    parent_labels: list[str] = []

    mapped_to_parent = defaultdict(set)      # "drug (mapping)" -> list of children

    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Raw split
        raw_ids = str(row[id_column]).split("|")
        raw_tokens = str(row[tokens_column]).split("|")
    
        # Filter based on ID != -1 while keeping alignment
        input_ids = []
        input_tokens = []
        for tid, tok in zip(raw_ids, raw_tokens):
            tid = tid.strip()
            tok = tok.strip()
            if not tid or tid == "-1":
                continue
            input_ids.append(tid)
            input_tokens.append(tok)
    
        row_parents: list[str] = []
        row_labels: list[str] = []
    
        # Loop over aligned ids + tokens
        for child_token, child_id in zip(input_tokens, input_ids):
            parents = get_all_mappings_with_labels(child_id, cui2_to_cui1, cui_to_str)  # list[(parent_id, parent_label)]
            if not parents:
                continue
    
            for parent_id, parent_label in parents:
                # Skip if parent not in counts (defensive) or too many children
                if parent_id not in parent_counts:
                    continue
                if parent_counts[parent_id] > 20:
                    continue
    
                if parent_id in all_umls_ids:
                    row_parents.append(parent_id)
                    row_labels.append(parent_label)
    
                    # Use the *token* for the child, not the ID
                    key = f"{parent_label} ({parent_id})"
                    mapped_to_parent[key].add(child_token)
                        
            
        parent_ids.append("|".join(row_parents) if row_parents else "-1")
        parent_labels.append("|".join(row_labels) if row_labels else "-1")
    
    # Attach results to a copy of the DataFrame
    result = df.copy()
    result["nearest_dataset_parent_umls"] = parent_ids
    result["nearest_dataset_parent_umls_label"] = parent_labels

    return result, mapped_to_parent

In [87]:
parent_counts['C0393022']

15

In [88]:
df_expanded_clinical, mapped_to_parent_clinical = assign_nearest_dataset_parents(
    df_clinical,
    cui2_to_cui1,
    cui_to_str,
    all_umls_ids,
    parent_counts,
    id_column = "drug_umls_termid"
)

In [89]:
df_expanded_preclinical, mapped_to_parent_preclinical = assign_nearest_dataset_parents(
    df_preclinical,
    cui2_to_cui1,
    cui_to_str,
    all_umls_ids,
    parent_counts,
    id_column = "drug_umls_termid"
)

In [90]:
mapped_to_parent_preclinical['riTUXimab (C0393022)']

{'131I-rituximab',
 '177Lu-DOTA-rituximab',
 '89Zr-rituximab',
 'technetium 99m carbonyl DTPA-rituximab',
 'technetium 99m rituximab',
 'thorium-227-DOTA-rituximab'}

In [91]:
mapped_to_parent_df = (
    pd.DataFrame([
        {
            "drug_mapping": key,
            "drug": key.split("(")[0],
            "mapping": key.split("(")[1].rstrip(")"),
            "children_values": values,
            "ner_count": len(values)
        }
        for key, values in mapped_to_parent_clinical.items()
    ])
)

# Optional: sort by count descending
mapped_to_parent_df = mapped_to_parent_df.sort_values(by="ner_count", ascending=False).reset_index(drop=True)
mapped_to_parent_df

,drug_mapping,drug,mapping,children_values,ner_count
0,Botulinum Toxin Type A (C0006050),Botulinum Toxin Type A,C0006050,"{onabotulinumtoxinA, AbobotulinumtoxinA, onabo...",7
1,"Vaccines, Combined (C0206253)","Vaccines, Combined",C0206253,"{Corynebacterium diphtheriae, measles, mumps, ...",7
2,"Vaccine, Meningococcal (C0700144)","Vaccine, Meningococcal",C0700144,"{MenAfriVac, MenACWY, tetravalent meningococca...",6
3,POLIOVIRUS VACCINE INACTIVATED (C0718003),POLIOVIRUS VACCINE INACTIVATED,C0718003,"{Corynebacterium diphtheriae, diphtheria-tetan...",6
4,Oxygen (C0030054),Oxygen,C0030054,"{Entonox, Oxygen, gas hypoxic mixture (nitroge...",5
...,...,...,...,...,...
345,(-)-Pramipexole (C0074710),,-)-Pramipexole,{Pramipexole dihydrochloride},1
346,"Benzoate, Sodium (C0142805)","Benzoate, Sodium",C0142805,{Benzoate},1
347,Solifenacin (C1099677),Solifenacin,C1099677,"{Succinate, Solifenacin}",1
348,Carbon (C0007009),Carbon,C0007009,{AST 120},1


In [92]:
mapped_to_parent_preclin_df = (
    pd.DataFrame([
        {
            "drug_mapping": key,
            "drug": key.split("(")[0],
            "mapping": key.split("(")[1].rstrip(")"),
            "children_values": values,
            "ner_count": len(values)
        }
        for key, values in mapped_to_parent_preclinical.items()
    ])
)

# Optional: sort by count descending
mapped_to_parent_preclin_df = mapped_to_parent_preclin_df.sort_values(by="ner_count", ascending=False).reset_index(drop=True)
mapped_to_parent_preclin_df

,drug_mapping,drug,mapping,children_values,ner_count
0,Gamma-aminobutyrate (C0178649),Gamma-aminobutyrate,C0178649,"{GABA-stearamide, progabide, 4-(2-(2,6-dimethy...",26
1,Valproate (C0080356),Valproate,C0080356,"{Propylisopropylactic Acid, Magnesium Valproat...",18
2,Eicosapentaenoate (C1370123),Eicosapentaenoate,C1370123,"{5S,12R,18R-trihydroxy-6Z,8E,10E,14Z,16E-eicos...",17
3,Quinate (C0524704),Quinate,C0524704,"{3,5-dicaffeoyl-4-malonylquinic acid, caffeoyl...",16
4,Hyaluronate (C0178695),Hyaluronate,C0178695,"{Acid, Hyaluronic, hylan, poly(gamma-benzyl gl...",15
...,...,...,...,...,...
1811,RNA Binding Protein FUS (C0950472),RNA Binding Protein FUS,C0950472,"{fused in sarcoma protein, human}",1
1812,Interleukin-7 (C0021761),Interleukin-7,C0021761,{efineptakin alfa},1
1813,11-beta-Hydroxysteroid Dehydrogenase Type 1 (C...,11-beta-Hydroxysteroid Dehydrogenase Type 1,C0909868,"{HSD11B1 protein, human}",1
1814,"Necrosis Factors, Tumor (C0041368)","Necrosis Factors, Tumor",C0041368,{activation-induced tumor necrosis factor liga...,1


In [93]:
df_expanded_clinical[df_expanded_clinical['nearest_dataset_parent_umls']!="-1"].head()

,nct_id,linkbert_aact_mapped_conditions,linkbert_aact_mapped_drugs,Disease Class,drug_term_umls_norm,drug_umls_termid,disease_term_mondo_norm,disease_mondo_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean,nearest_dataset_parent_umls,nearest_dataset_parent_umls_label
3,NCT00000439,alcohol dependence|bipolar alcoholics|bipolar ...,depacon|sodium valproate|valproate|valproic acid,Psychiatry and Psychology Category,depacon|Valproate Sodium|Valproate|Valproic Acid,-1|C0037567|C0080356|C0042291,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,-1|-1,-1|-1,MONDO:0002046|-1|MONDO:0004985|-1,alcohol abuse|bipolar alcoholics|bipolar disor...,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,C0080356|C0080356,Valproate|Valproate
14,NCT00005850,anxiety|anxiety disorders|chemotherapy|depress...,cisplatin|fluoxetine|gemcitabine hydrochloride...,Psychiatry and Psychology Category,"Cisplatin|Fluoxetine|Hydrochloride, Gemcitabin...",C0008838|C0016365|C0771488|-1,anxiety disorder|chemotherapy|depressive disor...,MONDO:0005618|-1|MONDO:0002050|-1|MONDO:000523...,-1|-1|MONDO:0003274,-1|-1|thoracic cancer,MONDO:0005618|-1|MONDO:0002050|-1|MONDO:000523...,anxiety disorder|chemotherapy|depressive disor...,anxiety disorder|chemotherapy|depressive disor...,MONDO:0005618|-1|MONDO:0002050|-1|MONDO:000523...,C0045093,Gemcitabine
21,NCT00006180,bone|depressed|depression|low|major depression...,alendronate|alendronic acid|calcium|ergocalcif...,Psychiatry and Psychology Category,Alendronate|Alendronic acid|Calcium|Ergocalcif...,C0102118|C0771311|C0006675|C0014695|C3714503,bone|depressive disorder|low|major depressive ...,-1|MONDO:0002050|-1|MONDO:0002009|MONDO:0005298,-1|MONDO:0002050|MONDO:0019019,-1|depressive disorder|osteogenesis imperfecta,-1|MONDO:0002050|-1|MONDO:0002009|MONDO:000529...,bone|depressive disorder|low|major depressive ...,bone|depressive disorder|low|major depressive ...,-1|MONDO:0002050|-1|MONDO:0002009|MONDO:000529...,C0102118,Alendronate
25,NCT00006349,brain metastases|cancer|cognitive dysfunction|...,donepezil|donepezil hydrochloride|vitamin e,Diseases of the nervous system|Neurologic Mani...,donepezil|Donepezil hydrochloride|Vitamin E,C0527316|C0771848|C0042874,metastatic malignant neoplasm in the brain|can...,MONDO:1040026|MONDO:0004992|MONDO:0002039|MOND...,MONDO:0024883|-1|-1|-1|-1|MONDO:0003274,metastatic neoplasm|-1|-1|-1|-1|thoracic cancer,MONDO:1040026|MONDO:0004992|MONDO:0002039|MOND...,metastatic malignant neoplasm in the brain|can...,metastatic malignant neoplasm in the brain|can...,MONDO:1040026|MONDO:0004992|MONDO:0002039|MOND...,C0527316,donepezil
26,NCT00006450,brain neoplasms|cancer|ependymoma|malignant ge...,phenylacetate|phenylbutyric acid,Nervous System Neoplasms,Phenylacetate|γ-phenylbutyric acid,C0220893|C0170531,brain neoplasm|cancer|childhood ependymoma|mal...,MONDO:0021211|MONDO:0004992|MONDO:0003478|MOND...,-1|-1|MONDO:0021042|MONDO:0005040|MONDO:000242...,-1|-1|glioma|germ cell tumor|cerebellar disord...,MONDO:0021211|MONDO:0004992|MONDO:0003478|MOND...,brain neoplasm|cancer|childhood ependymoma|mal...,brain neoplasm|cancer|childhood ependymoma|mal...,MONDO:0021211|MONDO:0004992|MONDO:0003478|MOND...,C0281398,Phenylbutyrate


## Merge with existing mappings, keep unique new terms

In [101]:
def merge_original_and_parent(
    df: pd.DataFrame,
    id_col: str = "disease_mondo_termid",
    label_col: str = "disease_term_mondo_norm",
    parent_id_col: str = "nearest_dataset_parent_mondo",
    parent_label_col: str = "nearest_dataset_parent_label",
    merged_id_col: str = "merged_mondo_termid",
    merged_label_col: str = "merged_mondo_label"
) -> pd.DataFrame:
    """
    For each row:
      1. Split `id_col` and `label_col` into parallel lists orig_ids, orig_labels.
      2. Split parent columns into parent_ids, parent_labels.
      3. Keep orig_ids & orig_labels exactly (including '-1' slots).
      4. Append any parent_id != '-1' that isn’t already in orig_ids, 
         along with its matching parent_label.
      5. Re-join into pipe-delimited merged_id_col / merged_label_col.

    Returns the modified DataFrame, and prints how many new labels were added.
    """

    df = df.copy()
    merged_ids = []
    merged_labels = []

    added_count = 0   # count new parent labels added

    for _, row in df.iterrows():
        # 1) Original
        orig_ids    = row[id_col].split("|")
        orig_labels = row[label_col].split("|")

        # 2) Parents
        parent_ids    = row[parent_id_col].split("|")
        parent_labels = row[parent_label_col].split("|")

        # 3) Start with originals in order
        mids  = list(orig_ids)
        mlabs = list(orig_labels)

        # 4) Append parents if new (keeps order)
        for pid, plab in zip(parent_ids, parent_labels):
            if pid != "-1" and pid not in mids:
                mids.append(pid)
                mlabs.append(plab)
                added_count += 1

        # ---- ORDER-PRESERVING DEDUPLICATION ----
        seen = set()
        final_ids = []
        final_labels = []

        for mid, mlab in zip(mids, mlabs):
            if mid not in seen:
                seen.add(mid)
                final_ids.append(mid)
                final_labels.append(mlab)

        # 5) Join back
        merged_ids.append("|".join(final_ids))
        merged_labels.append("|".join(final_labels))

    df[merged_id_col]    = merged_ids
    df[merged_label_col] = merged_labels

    print(f"Total new parent labels added: {added_count}")

    return df


In [102]:
df_final_preclinical = merge_original_and_parent(
    df_expanded_preclinical,
    id_col="drug_umls_termid",
    label_col="drug_term_umls_norm",
    parent_id_col="nearest_dataset_parent_umls",
    parent_label_col="nearest_dataset_parent_umls_label",
    merged_id_col= "merged_umls_termid",
    merged_label_col= "merged_umls_label"
)

Total new parent labels added: 28920


In [103]:
df_final_preclinical.head()

,PMID,unique_conditions_linkbert_predictions,unique_interventions_linkbert_predictions,linkbert_mapped_conditions,linkbert_mapped_drugs,disease_term_mondo_norm,disease_mondo_termid,drug_term_umls_norm,drug_umls_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean,nearest_dataset_parent_umls,nearest_dataset_parent_umls_label,merged_umls_termid,merged_umls_label
0,31733831,asthma,isorhynchophylline,asthma,isorhynchophylline,asthma,MONDO:0004979,isorhynchophylline,C0245133,-1,-1,MONDO:0004979,asthma,asthma,MONDO:0004979,-1,-1,C0245133,isorhynchophylline
1,31733833,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,antgomir-1192|mir-1192|agomir-1192,myocardial infarction,MONDO:0005068,antgomir-1192|mir-1192|agomir-1192,-1|-1|-1,-1,-1,MONDO:0005068,myocardial infarction,myocardial infarction,MONDO:0005068,-1,-1,-1,antgomir-1192
2,31733925,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,g2|hla-g2,systemic lupus erythematosus,MONDO:0007915,g2|HLA-G2 Isoform,-1|C0967254,-1,-1,MONDO:0007915,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915,-1,-1,-1|C0967254,g2|HLA-G2 Isoform
3,31733940,cognitive impairment,minocycline,cognitive impairment,minocycline,cognitive disorder,MONDO:0002039,Minocycline,C0026187,-1,-1,MONDO:0002039,cognitive disorder,cognitive disorder,MONDO:0002039,-1,-1,C0026187,Minocycline
4,31734027,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,tadalafil|phosphodiesterase type 5 inhibitor t...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,Tadalafil|Tadalafil,C1176316|C1176316,-1,-1,-1|-1|MONDO:0003620,oxaliplatin-induced peripheral neuropathy|cumu...,oxaliplatin-induced peripheral neuropathy|cumu...,-1|-1|MONDO:0003620,-1,-1,C1176316,Tadalafil


In [104]:
df_final_clinical = merge_original_and_parent(
    df_expanded_clinical,
    id_col="drug_umls_termid",
    label_col="drug_term_umls_norm",
    parent_id_col="nearest_dataset_parent_umls",
    parent_label_col="nearest_dataset_parent_umls_label",
    merged_id_col= "merged_umls_termid",
    merged_label_col= "merged_umls_label"
)

Total new parent labels added: 1183


In [105]:
df_final_clinical.head()

,nct_id,linkbert_aact_mapped_conditions,linkbert_aact_mapped_drugs,Disease Class,drug_term_umls_norm,drug_umls_termid,disease_term_mondo_norm,disease_mondo_termid,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label,disease_term_mondo_parent_clean,disease_termid_mondo_parent_clean,nearest_dataset_parent_umls,nearest_dataset_parent_umls_label,merged_umls_termid,merged_umls_label
0,NCT00000307,alcohol-related disorders|alcoholic cocaine de...,naltrexone,Diseases Category,Naltrexone,C0027360,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186,-1|MONDO:0005303,-1|drug dependence,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303,alcohol-related disorders|alcoholic cocaine de...,alcohol-related disorders|alcoholic cocaine de...,MONDO:0021698|-1|MONDO:0005186|MONDO:0005303,-1,-1,C0027360,Naltrexone
1,NCT00000333,cocaine|cocaine-related disorders,atropine|benzatropine|benztropine|da reuptake ...,Diseases Category,Atropine|Benztropine|Benztropine|da reuptake i...,C0004259|C0005098|C0005098|-1,cocaine|cocaine dependence,-1|MONDO:0005186,MONDO:0005303,drug dependence,-1|MONDO:0005186|MONDO:0005303,cocaine|cocaine dependence|drug dependence,cocaine|cocaine dependence|drug dependence,-1|MONDO:0005186|MONDO:0005303,-1,-1,C0004259|C0005098|-1,Atropine|Benztropine|da reuptake inhibitor
2,NCT00000428,fibromyalgia,amitriptyline|amitriptyline plus fluoxitine|fl...,Neuromuscular Diseases,Amitriptyline|amitriptyline plus fluoxitine|fl...,C0002600|-1|-1|C0016365,fibromyalgia,MONDO:0005546,-1,-1,MONDO:0005546,fibromyalgia,fibromyalgia,MONDO:0005546,-1,-1,C0002600|-1|C0016365,Amitriptyline|amitriptyline plus fluoxitine|Fl...
3,NCT00000439,alcohol dependence|bipolar alcoholics|bipolar ...,depacon|sodium valproate|valproate|valproic acid,Psychiatry and Psychology Category,depacon|Valproate Sodium|Valproate|Valproic Acid,-1|C0037567|C0080356|C0042291,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,-1|-1,-1|-1,MONDO:0002046|-1|MONDO:0004985|-1,alcohol abuse|bipolar alcoholics|bipolar disor...,alcohol abuse|bipolar alcoholics|bipolar disor...,MONDO:0002046|-1|MONDO:0004985|-1,C0080356|C0080356,Valproate|Valproate,-1|C0037567|C0080356|C0042291,depacon|Valproate Sodium|Valproate|Valproic Acid
4,NCT00001956,hyperalgesia|oral surgery|pain|removal of thir...,|ketorolac|lidocaine|midazolam|sedative,Neurologic Manifestations,Ketorolac|Lidocaine|Midazolam|sedative,C0073631|C0023660|C0026056|-1,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1,-1,-1,-1|-1|MONDO:0021668|-1,hyperalgesia|oral surgery|obsolete disorder in...,hyperalgesia|oral surgery|obsolete disorder in...,-1|-1|MONDO:0021668|-1,-1,-1,C0073631|C0023660|C0026056|-1,Ketorolac|Lidocaine|Midazolam|sedative


In [106]:
df_final_preclinical.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_preclinical_data_with_umls_mondo_parents_mondo_clean.csv", index=False)
df_final_clinical.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_clinical_data_with_umls_mondo_parents_mondo_clean.csv", index=False)
